**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/sabatini-2024"], check=True)
    os.chdir("/content/encore/entries/sabatini-2024")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# The earlier attempt's findings, checked on DANDI:000070 version 0.261122.2046

Attempt `sabatini-2024-02` was made blind on version 0.261122.2046. The attempt before it,
`sabatini-2024-01`, was made on version 0.260528.0919 and recorded eight findings. This notebook
checks, on version 0.261122.2046, each earlier finding that the blind attempt did not report
(docs/protocol.md, "Repeat attempts"), one section per finding, each opening with the earlier
statement. It reads the two files the paper names, through the entry's helper `ldr.py`
(`ldr.VERSION`), and loads only metadata, timestamps and spike times.

In [ ]:
import numpy as np
import ldr

print("dandiset", ldr.DANDISET, "version", ldr.VERSION)
H = {m: ldr.open_nwb(ldr.SESSIONS[m]) for m in ("J", "N")}
SP = {}
for m, h in H.items():
    idx = h["units/spike_times_index"][:]
    SP[m] = (np.r_[0, idx[:-1]], idx)
    t = h["intervals/trials"]
    print(m, ldr.SESSIONS[m])
    print("   trials", len(t["start_time"]), "| successful", int((t["task_success"][:] == 1).sum()),
          "| units", len(idx), "| spike times", int(idx[-1]),
          "| hand samples", h["processing/behavior/Position/Hand/timestamps"].shape[0])

## Earlier finding 1: the Nitschke file's overlapping clocks

Earlier statement (sabatini-2024-01, finding 1):

> The Nitschke file (sub-Nitschke_ses-20100923) joins five recording segments whose clocks overlap, and no metadata says so. Trial start times, hand timestamps and every unit's spike times jump backwards four times (hand timestamps span 5.4 to 6075.4 s, 6074.2 to 12075.9 s, 6070.6 to 12493.5 s, 6119.8 to 13515.2 s and 6073.5 to 6656.8 s), so spike times are not sorted and a trial's spikes cannot be selected by time alone; doing so mixes spikes of up to four segments. Splitting each unit's spikes at the jumps and assigning trials to segments in file order gives condition decoding of 0.44 to 0.50 in segments 0 to 2 and 0.18 in segment 3 (all successful trials; chance 0.014; 0.010 to 0.024 with the previous trial's labels).

In [ ]:
for m, h in H.items():
    st = h["intervals/trials/start_time"][:]
    ts = h["processing/behavior/Position/Hand/timestamps"][:]
    a, b = SP[m]
    sp = h["units/spike_times"]
    unsorted = [u for u in range(len(b)) if np.any(np.diff(sp[a[u]:b[u]]) < 0)]
    print(m, "trial starts going backwards:", int((np.diff(st) < 0).sum()),
          "| hand timestamps going backwards:", int((np.diff(ts) < 0).sum()),
          f"| hand timestamps {ts.min():.1f} to {ts.max():.1f} s",
          "| units with unsorted spike times:", len(unsorted))

Each file now has one clock: trial starts, hand timestamps and every unit's spike times only
go forwards, so a trial's spikes can be selected by time. That trials, spikes and hand share this
clock is the blind attempt's data check (notebook 1, cell `sanity`: hand speed peaks 120 to 140 ms
after the table's movement onset, and reach direction decodes from spikes after onset).

## Earlier finding 2: one unit per electrode, with no sorting label

Earlier statement (sabatini-2024-01, finding 2):

> The units table holds 96 units per array, exactly one on each of the 192 electrodes, in both files. Its columns are electrode_group, electrodes, id, obs_intervals and spike_times, with no column for single or multi-unit, sorting quality or isolation. The paper reports 79 to 118 single and stable multi-units per array. Whether these units are threshold crossings or sorted units merged per electrode is not stated.

In [ ]:
for m, h in H.items():
    print(m, "units description:", h["units"].attrs["description"])
    print("   columns:", sorted(h["units"].keys()))
    el = h["units/electrodes"][:]
    print("   units", len(el), "on", len(np.unique(el)), "electrodes")

The blind attempt reports this as its finding 1. The units table now says what the units are
(unsorted threshold crossings, one per electrode); the sorted units the paper used are still not
in the files.

## Earlier finding 3: Jenkins unit 1's row holding two spike trains

Earlier statement (sabatini-2024-01, finding 3):

> In the Jenkins file (sub-Jenkins_ses-20090918), unit 1's row of the units table holds two sorted spike trains one after the other. The first, 196608 spikes from 4259.1 to 15668.6 s, begins 55 ms after unit 0's train, which holds 61030 spikes from 4.9 s, stops at 4259.0 s. The second, 264946 spikes from 4.7 to 15668.6 s, spans the session as the other units do. The two share 1049 spike times; neither has intervals under 1 ms, while merged 3.5% of intervals are. Of the first run's times 37.4% are found exactly in another unit's train (unit 68), as are 27.9% of unit 0's, against at most 2.5% for the second run. Unit 1's row thus appears to hold the rest of unit 0's train after 4259 s, and unit 0's row is cut short. Nothing but the unsorted order of unit 1's spike times shows it; merging the two runs, as the main analyses do, gives unit 1 a train with another unit's spikes. The companion notebook leaves units 0 and 1 out.

In [ ]:
h = H["J"]; a, b = SP["J"]
st = h["intervals/trials/start_time"][:]
for u in (0, 1):
    x = h["units/spike_times"][a[u]:b[u]]
    d = np.diff(x)
    # mean rate over trials in the first and second half of the session
    c = np.searchsorted(x, h["intervals/trials/stop_time"][:]) - np.searchsorted(x, st)
    r = c / (h["intervals/trials/stop_time"][:] - st)
    half = len(st) // 2
    print(f"unit {u}: {len(x)} spikes, {x[0]:.1f} to {x[-1]:.1f} s, times going backwards {int((d < 0).sum())},"
          f" intervals under 1 ms {np.mean(d < 1e-3):.4f}, mean rate in trials of the first and second half"
          f" {r[:half].mean():.1f} and {r[half:].mean():.1f} spikes/s")
x0 = h["units/spike_times"][a[0]:b[0]]; x1 = h["units/spike_times"][a[1]:b[1]]
print("spike times shared by units 0 and 1:", len(np.intersect1d(x0, x1)))

Unit 0 now spans the whole session and unit 1's spike times are sorted. In the earlier
version unit 0 held 61030 spikes and the first run of unit 1 196608; unit 0 now holds their sum,
and unit 1 holds as many spikes as the earlier second run, 264946.

## Earlier finding 4: positions labelled meters with a conversion of NaN

Earlier statement (sabatini-2024-01, finding 4):

> Hand, cursor and eye positions are labelled "meters" with a conversion of NaN, but hold millimetres (reaches of about 130 and positions from -182 to 159).

In [ ]:
for m, h in H.items():
    for k in ("Hand", "Cursor", "Eye"):
        d = h[f"processing/behavior/Position/{k}/data"]
        v = d[:100000]
        print(m, k, "| unit", d.attrs["unit"], "| conversion", d.attrs["conversion"],
              f"| stored values {np.nanmin(v):.1f} to {np.nanmax(v):.1f} over the first 100,000 samples")

Hand and cursor positions now carry a conversion of 0.001, so their millimetres convert to
meters. Eye position still carries a conversion of NaN under the unit meters, with stored values in
the hundreds.

## Earlier finding 5: no EMG

Earlier statement (sabatini-2024-01, finding 5):

> The dataset holds no EMG, which the paper's Supplementary Figure 8 removes along with kinematic dimensions.

In [ ]:
for m, h in H.items():
    names = []
    h.visit(lambda n: names.append(n) if ("emg" in n.lower() or "muscle" in n.lower()) else None)
    print(m, "acquisition:", sorted(h["acquisition"].keys()) if "acquisition" in h else "none",
          "| processing:", {k: sorted(h["processing"][k].keys()) for k in h["processing"]})
    print("   paths naming EMG or muscle:", names)

Neither file holds EMG: there is no acquisition series, the processing modules hold positions
and filtered ephys, and no object is named for EMG or muscle.

## Earlier finding 6: Nitschke unit 170's rate in one segment

Earlier statement (sabatini-2024-01, finding 6):

> In the Nitschke file, one M1 unit (unit 170) fires at a mean of 175.2 spikes/s in segment 3, above 100 spikes/s on 229 of its 532 trials (trials starting 6446 to 10006 s on that segment's clock), against means of 27.6 to 40.4 spikes/s and no such trial in the other segments. Its ratio of largest to median mean rate over blocks of 50 trials is 13.7, against at most 2.5 for the other units of the file. Nothing in the file marks it. It lowers condition decoding within segment 3 to 0.179, against 0.445 to 0.495 in segments 0 to 2; without it, decoding within segment 3 is 0.425. The per-trial rates summed over units have similar ranges in all segments. The companion notebook leaves this unit out.

In [ ]:
h = H["N"]; a, b = SP["N"]
d = ldr.trial_table(h)
sel = ldr.select_trials(d) & np.isfinite(d.move_begins_time.values)
on = d.move_begins_time.values[sel]
t0, t1 = on - ldr.PRE_MS / 1000, on + ldr.POST_MS / 1000
sp = h["units/spike_times"]
rate = np.zeros((len(on), len(b)))
for u in range(len(b)):
    x = sp[a[u]:b[u]]
    rate[:, u] = (np.searchsorted(x, t1) - np.searchsorted(x, t0)) / (t1 - t0)
# blocks of 50 trials in file order: largest block mean over median block mean (as the earlier attempt)
nb = len(on) // 50
B = np.array([rate[i * 50:(i + 1) * 50].mean(0) for i in range(nb)])
ratio = B.max(0) / np.median(B, 0)
hi = rate[:, 170] > 100
print(len(on), "successful trials of the 72 conditions, -350 to +850 ms around move_begins_time")
print(f"unit 170: mean {rate[:, 170].mean():.1f} spikes/s; above 100 spikes/s on {int(hi.sum())} trials,"
      f" starting {d.start_time.values[sel][hi].min():.0f} to {d.start_time.values[sel][hi].max():.0f} s;"
      f" mean on those {rate[hi, 170].mean():.1f}, on the others {rate[~hi, 170].mean():.1f} spikes/s")
others = np.delete(ratio, 170)
print(f"block ratio, unit 170: {ratio[170]:.1f}; other units: at most {np.nanmax(others[np.isfinite(others)]):.1f}"
      f" ({int(np.sum(~np.isfinite(others)))} units silent in most blocks)")

Unit 170 still fires above 100 spikes/s on a run of trials in one part of the session, with
a block ratio far above every other unit's, and nothing in the units table marks it.

## Earlier finding 7: questions for the paper's authors

Earlier statement (sabatini-2024-01, finding 7):

> Questions for the paper's authors. (1) Which trials were analysed? All successful trials give 1595 (J) and 2121 (N), and successful trials with a delay above 400 ms 1381 and 1884, against 1469 and 2014 in the Figure 7 legend; the first was used here, and the second gives the same outcomes. (2) Which units were used, and are they in the dandiset? The files hold one unit per electrode with no single or multi-unit label. (3) What penalty was used for the OU kernel encoding models with length scales 1e11 and 1e12? With a penalty of 0.1 they predict only the mean; none was used here. (4) How were the noise excursion angles of Figure 4c computed? (5) Were the jPCA and single-condition LDS values of Supplementary Figure 3 cross-validated, and how? (6) Did the trial matrices shuffled in the control for data smoothing span the analysis window, as preprocessTrials.m implies? (7) What scale do the kinematic basis functions (kinFuncEval) have, and in what units are rates given to the OU kernel decoders with length scale 100, and on how many points are they trained?

This finding is about the paper, which has not changed, and not about the files. The blind
attempt's own questions (its finding 5) cover the size of the training sets of the kernel decoders,
and its finding 1 settles what the units are; the other questions stay open and are added to the
record as a finding found on prompt.

## Earlier finding 8: the paper's code needs files available only on request

Earlier statement (sabatini-2024-01, finding 8):

> The paper's code (kaufmanlab/LDR-public) loads preprocessed files (ShenoyMonkeyData and others) that the Data availability statement offers only on request, so the code cannot be run on the public data, and the trial and unit selection behind them cannot be checked.

The blind attempt reports this as its finding 6, which names the same preprocessed .mat files
available only on request. Nothing to run here.